[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andreas-he/ai-safety-challenges/blob/main/challenges/2026-04-16-logit-difference/challenge.ipynb)

# Logit Difference — the Workhorse Mech-Interp Metric

**Category:** Mech Interp &middot; **Format:** short-coding &middot; **Est:** 25 min

**Relevance.** Logit difference — `logits[correct] - logits[incorrect]` — is the single most-used metric in circuit analysis (IOI, activation patching, attribution patching, direct logit attribution). It's a linear, per-token, bounded-below-by--inf signal that cleanly factorises through the residual stream. Get fluent with it here — SAIGE/LASR mech-interp work uses it constantly.

You'll implement three tiny utilities:

1. `logit_diff(logits, correct, incorrect)` — scalar for one prediction
2. `logit_diff_batch(logits, correct_idx, incorrect_idx)` — vector over a batch
3. `patching_score(clean_ld, corrupted_ld, patched_ld)` — normalised 0-to-1 activation-patching metric

All pure numpy. No transformers library needed.


## Setup


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
VOCAB = 50_000


## Task 1 — Single-example logit difference

Given a 1-D logits vector of length `VOCAB` and two token ids (correct and incorrect), return `logits[correct] - logits[incorrect]` as a Python float.

> Why this metric and not log-softmax difference? The softmax's normaliser (logsumexp) is shared across tokens, so it cancels in the difference of log-probs. Logit diff is therefore (up to a positive constant) the same signal without the compute.


In [ ]:
def logit_diff(logits: np.ndarray, correct: int, incorrect: int) -> float:
    """
    Args:
        logits: shape (VOCAB,)
        correct, incorrect: token ids
    Returns:
        float — logits[correct] - logits[incorrect]
    """
    raise NotImplementedError


### Tests — run this cell; all asserts should pass.


In [ ]:
logits = np.zeros(VOCAB); logits[7] = 3.0; logits[42] = 1.5
assert abs(logit_diff(logits, 7, 42) - 1.5) < 1e-9, 'scalar case failed'
assert abs(logit_diff(logits, 42, 7) + 1.5) < 1e-9, 'swap should negate'
assert logit_diff(logits, 7, 7) == 0.0, 'correct == incorrect must be 0'
print('Task 1 OK')


## Task 2 — Batched logit difference

Now vectorise: given `logits` of shape `(B, VOCAB)` and integer arrays `correct_idx`, `incorrect_idx` of shape `(B,)`, return a `(B,)` array of per-example logit differences. **No Python `for` loop.**

> This is the pattern used in IOI: batch over ~300 prompts, each with its own (IO, S) token pair, and track how logit diff moves when you patch activations.


In [ ]:
def logit_diff_batch(logits: np.ndarray, correct_idx: np.ndarray, incorrect_idx: np.ndarray) -> np.ndarray:
    """
    Args:
        logits: shape (B, VOCAB)
        correct_idx, incorrect_idx: shape (B,) int
    Returns:
        shape (B,) — per-example logit diff
    """
    raise NotImplementedError


### Tests


In [ ]:
B = 4
logits_b = rng.standard_normal((B, VOCAB))
correct = np.array([10, 20, 30, 40])
incorrect = np.array([11, 21, 31, 41])
out = logit_diff_batch(logits_b, correct, incorrect)
assert out.shape == (B,), f'bad shape {out.shape}'
# Compare to naive loop
expected = np.array([logits_b[i, correct[i]] - logits_b[i, incorrect[i]] for i in range(B)])
assert np.allclose(out, expected), 'batched != per-row loop'
print('Task 2 OK')


## Task 3 — Activation-patching score (normalised)

In activation patching you get three numbers per example:

- `clean_ld` — logit diff when the model runs on the *clean* prompt (correct answer wins: ~large positive)
- `corrupted_ld` — logit diff on the *corrupted* prompt (correct answer suppressed: near 0 or negative)
- `patched_ld` — corrupted run *with a single component replaced by its clean activation* — measures how much of the clean behaviour that component causally restores

Define the **normalised patching score**:

$$\\text{score} = \\frac{\\text{patched\\_ld} - \\text{corrupted\\_ld}}{\\text{clean\\_ld} - \\text{corrupted\\_ld}}$$

- 0 → patching that component changed nothing (component is not causally relevant)
- 1 → patching fully restored clean behaviour (component is *sufficient*)
- between → partial contribution

Return NaN if `clean_ld == corrupted_ld` (degenerate — nothing to normalise by).


In [ ]:
def patching_score(clean_ld: float, corrupted_ld: float, patched_ld: float) -> float:
    """
    Normalised activation-patching score. NaN if clean == corrupted.
    """
    raise NotImplementedError


### Tests


In [ ]:
assert abs(patching_score(4.0, 0.0, 0.0) - 0.0) < 1e-9, 'patched==corrupted → 0'
assert abs(patching_score(4.0, 0.0, 4.0) - 1.0) < 1e-9, 'patched==clean → 1'
assert abs(patching_score(4.0, 0.0, 2.0) - 0.5) < 1e-9, 'halfway'
assert abs(patching_score(4.0, -2.0, 1.0) - 0.5) < 1e-9, 'negative corrupted baseline'
s = patching_score(1.0, 1.0, 1.0)
assert s != s, 'degenerate case must be NaN'
print('Task 3 OK')


## Reflection (1 min)

When you see a patching-score heatmap over (layer, position), you're looking at this function evaluated for each component. A spot of 0.8 at (layer=8, position=12) means: patching the layer-8 residual stream at the IO token restored 80% of the clean-vs-corrupted logit gap on average. That's causal evidence for the circuit.


---

<details>
<summary><b>Solution — click to reveal when you're done (or stuck)</b></summary>

```python
def logit_diff(logits, correct, incorrect):
    return float(logits[correct] - logits[incorrect])

def logit_diff_batch(logits, correct_idx, incorrect_idx):
    B = logits.shape[0]
    rows = np.arange(B)
    return logits[rows, correct_idx] - logits[rows, incorrect_idx]

def patching_score(clean_ld, corrupted_ld, patched_ld):
    denom = clean_ld - corrupted_ld
    if denom == 0:
        return float('nan')
    return (patched_ld - corrupted_ld) / denom
```

**Key insight.** All three functions are *linear* in the logits. That linearity is why activation patching decomposes cleanly through the residual stream via direct logit attribution — each component's contribution to logit diff is just the dot product of its output with the (unembed[correct] − unembed[incorrect]) direction. This is the mathematical backbone of most published circuit-analysis results.

</details>
